# Potomac flow-barrier spinup — total storage trends

Compare domain-integrated **total storage** (subsurface + surface ponding via `parflow.tools.hydrology`) across flow-barrier spinup runs:

- `potomac_flow_barrier_002`
- `potomac_flow_barrier_005`
- `potomac_flow_barrier_0011`
- `potomac_flow_barrier_0015`

Reads annual ParFlow dumps in each domain's `spinup/run` directory (`run.out.press.#####.pfb` / `run.out.satur.#####.pfb`, `DumpInterval = 8760`).

Use a Jupyter kernel with `parflow` and `numpy` (same environment as `do_spinup.py` / `steady_state_spinup.py`).

In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import parflow as pf
from parflow.tools.hydrology import (
    calculate_subsurface_storage,
    calculate_surface_storage,
)

PROJECT_ROOT = Path("/glade/derecho/scratch/bwest/drought-ensemble")
DOMAINS = [
    "potomac_flow_barrier_002",
    "potomac_flow_barrier_005",
    "potomac_flow_barrier_0011",
    "potomac_flow_barrier_0015",
]

DX = 1000.0
DY = 1000.0
DZ_BASE = 200.0
DZ = (
    np.array([1.0, 0.5, 0.25, 0.125, 0.05, 0.025, 0.005, 0.003, 0.0015, 0.0005], dtype=np.float64)
    * DZ_BASE
)
HOURS_PER_YEAR = 365.0 * 24.0

# Set to an integer to cap plots (e.g. 20 for early spinup); None uses all available annual dumps.
MAX_YEAR = None

In [ ]:
PRESS_TAG_RE = re.compile(r"run\.out\.press\.(\d+)\.pfb$")


def run_dir_for_domain(domain_name: str) -> Path:
    return PROJECT_ROOT / "domains" / domain_name / "spinup" / "run"


def discover_spinup_years(run_dir: Path) -> np.ndarray:
    years = []
    for path in sorted(run_dir.glob("run.out.press.*.pfb")):
        match = PRESS_TAG_RE.match(path.name)
        if not match:
            continue
        year = int(match.group(1))
        if year >= 1:
            years.append(year)
    if not years:
        raise FileNotFoundError(f"No annual pressure outputs in {run_dir}")
    years = np.array(sorted(years), dtype=int)
    if MAX_YEAR is not None:
        years = years[years <= MAX_YEAR]
    return years


def integrated_total_storage_m3(pressure, saturation, mask, porosity, specific_storage):
    mask_bin = np.where(mask > 0, 1, 0)
    subs = calculate_subsurface_storage(
        porosity,
        pressure,
        saturation,
        specific_storage,
        DX,
        DY,
        DZ,
        mask=mask_bin,
    )
    surf = calculate_surface_storage(pressure, DX, DY, mask=mask_bin)
    return float(np.sum(subs) + np.sum(surf))


def yearly_pme_total_m3(run_dir: Path, mask) -> float:
    pme_path = run_dir / "pme.pfb"
    if not pme_path.is_file():
        raise FileNotFoundError(f"Missing {pme_path}")
    pme = pf.read_pfb(str(pme_path))
    mask_2d = mask[0, :, :] > 0
    if pme.ndim != 3:
        raise ValueError(f"Unexpected pme.pfb shape: {pme.shape}")
    layer_activity = np.sum(np.abs(pme), axis=(1, 2))
    active_layers = np.where(layer_activity > 0)[0]
    if active_layers.size == 1:
        pme_2d = pme[active_layers[0], :, :]
    else:
        pme_2d = np.sum(pme, axis=0)
    return float(np.sum(np.where(mask_2d, pme_2d, 0.0)) * DX * DY * HOURS_PER_YEAR)


def load_spinup_storage_series(domain_name: str) -> dict:
    run_dir = run_dir_for_domain(domain_name)
    years = discover_spinup_years(run_dir)

    mask = pf.read_pfb(str(run_dir / "run.out.mask.pfb"))
    porosity = pf.read_pfb(str(run_dir / "run.out.porosity.pfb"))
    specific_storage = pf.read_pfb(str(run_dir / "run.out.specific_storage.pfb"))

    totals_m3 = []
    for y in years:
        tag = f"{y:05d}"
        press_path = run_dir / f"run.out.press.{tag}.pfb"
        sat_path = run_dir / f"run.out.satur.{tag}.pfb"
        if not press_path.is_file():
            raise FileNotFoundError(press_path)
        if not sat_path.is_file():
            raise FileNotFoundError(sat_path)
        pressure = pf.read_pfb(str(press_path))
        saturation = pf.read_pfb(str(sat_path))
        totals_m3.append(
            integrated_total_storage_m3(
                pressure, saturation, mask, porosity, specific_storage
            )
        )

    totals_m3 = np.asarray(totals_m3, dtype=np.float64)
    slope, intercept = np.polyfit(years.astype(float), totals_m3, 1)
    trend_m3 = slope * years.astype(float) + intercept

    yearly_pme_m3 = yearly_pme_total_m3(run_dir, mask)
    years_delta = years[1:]
    delta_storage_m3 = np.diff(totals_m3)
    fraction_of_yearly_pme = delta_storage_m3 / yearly_pme_m3

    return {
        "domain": domain_name,
        "run_dir": run_dir,
        "years": years,
        "totals_m3": totals_m3,
        "slope_m3_per_year": slope,
        "trend_m3": trend_m3,
        "yearly_pme_m3": yearly_pme_m3,
        "years_delta": years_delta,
        "fraction_of_yearly_pme": fraction_of_yearly_pme,
    }

In [ ]:
results = {}
for domain in DOMAINS:
    print(f"Loading {domain} ...")
    results[domain] = load_spinup_storage_series(domain)
    r = results[domain]
    print(
        f"  years {r['years'][0]}–{r['years'][-1]} ({len(r['years'])} dumps), "
        f"trend = {r['slope_m3_per_year']:,.0f} m³/year, "
        f"yearly PME = {r['yearly_pme_m3']:,.3e} m³/year"
    )
    print()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=False)
axes = axes.ravel()

for ax, domain in zip(axes, DOMAINS):
    r = results[domain]
    ax.plot(r["years"], r["totals_m3"] / 1e6, "o", color="C0", ms=3, label="Annual total storage")
    ax.plot(r["years"], r["trend_m3"] / 1e6, "-", color="C1", lw=1.5, label="Linear trend")
    ax.set_title(domain.replace("potomac_flow_barrier_", "FBZ "))
    ax.set_xlabel("Spinup year (dump index)")
    ax.set_ylabel("Total storage (million m³)")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

fig.suptitle("Potomac flow-barrier spinup — domain-integrated total storage", y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=False)
axes = axes.ravel()

for ax, domain in zip(axes, DOMAINS):
    r = results[domain]
    ax.plot(
        r["years_delta"],
        r["fraction_of_yearly_pme"],
        "o-",
        color="C3",
        ms=3,
    )
    ax.axhline(0.0, color="k", lw=1)
    ax.set_title(domain.replace("potomac_flow_barrier_", "FBZ "))
    ax.set_xlabel("Spinup year")
    ax.set_ylabel(r"$\Delta S$ / yearly PME (-)")
    ax.grid(True, alpha=0.3)

fig.suptitle("Potomac flow-barrier spinup — annual storage change / yearly PME", y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
# Overlay: compare total storage trends on one axis (normalize to year 1)
fig, ax = plt.subplots(figsize=(9, 5))

for domain in DOMAINS:
    r = results[domain]
    s0 = r["totals_m3"][0]
    ax.plot(
        r["years"],
        (r["totals_m3"] - s0) / 1e6,
        "o-",
        ms=3,
        label=domain.replace("potomac_flow_barrier_", "FBZ "),
    )

ax.set_xlabel("Spinup year (dump index)")
ax.set_ylabel("Total storage change from year 1 (million m³)")
ax.set_title("Flow-barrier spinup — storage change relative to year 1")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()